# Sathi intent: BanglaBERT research model (offline, not deployed)

Fine-tunes `csebuetnlp/banglabert` on the **train** split of `eval/intent_dataset.csv` and reports per-class
recall with 95% Wilson intervals on the **human-written held-out** split, next to the deployed model
(`eval/run_intent_eval.py`: NVIDIA nemotron-3.5-embed-1b embeddings + logistic regression).

Rules: this model is never deployed and never decides anything. The held-out split is never generated by an
LLM. Run on Colab with a GPU (Runtime → Change runtime type → GPU). Upload `intent_dataset.csv` when asked.

In [ ]:
!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate==1.11.0"

In [ ]:
import csv, math, io
from google.colab import files
uploaded = files.upload()  # choose eval/intent_dataset.csv
rows = list(csv.DictReader(io.StringIO(next(iter(uploaded.values())).decode("utf-8"))))
labels = ["app", "medical", "emergency"]
train = [r for r in rows if r["split"] == "train"]
heldout = [r for r in rows if r["split"] == "heldout" and r["source"] != "synthetic"]
print(len(train), "train rows,", len(heldout), "human held-out rows")
assert len(train) >= 60, "add more training examples first (eval/add_intent_example.py)"

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
name = "csebuetnlp/banglabert"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=len(labels))
enc = lambda b: tok(b["text"], truncation=True, max_length=96)
ds = Dataset.from_list([{"text": r["text"], "label": labels.index(r["label"])} for r in train]).map(enc, batched=True)
args = TrainingArguments("banglabert-intent", num_train_epochs=4, per_device_train_batch_size=16,
                         learning_rate=3e-5, weight_decay=0.01, save_strategy="no", report_to=[], seed=7)
Trainer(model=model, args=args, train_dataset=ds, tokenizer=tok).train()

In [ ]:
import torch
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)
model.eval()
hits = {l: [0, 0] for l in labels}
for r in heldout:
    with torch.no_grad():
        logits = model(**{k: v.to(model.device) for k, v in tok(r["text"], return_tensors="pt", truncation=True).items()}).logits
    pred = labels[int(logits.argmax())]
    hits[r["label"]][1] += 1; hits[r["label"]][0] += int(pred == r["label"])
for l, (k, n) in hits.items():
    lo, hi = wilson(k, n)
    print(f"{l:10} recall {k}/{n} = {k/max(n,1):.0%} (95% CI {lo:.0%}-{hi:.0%})")

Copy the recall lines into `eval/intent_model_card.md` (research model column) next to the deployed model's
numbers from `eval/intent_results.json`. Do not ship this model: it is for comparison only.